<a href="https://colab.research.google.com/github/mihirtripathi97/protoplanetary-disk-analyzer/blob/main/Distance_Ladder_Project_B_Philhour_2026_(WITH_SOLUTIONS_ENTERED).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Distance Ladder Project - B. Philhour 2026

Note: the GAIA color-magnitude diagram material relies on the work of Richard Piccioni, Adjunct Faculty, Department of Physical Sciences, College of Marin, Acknowledgements: Irene Abril-Cabezas, Pythonista

Full reference here: https://ui.adsabs.harvard.edu/abs/2022ASPC..533..247P/abstract

Additional code for the Main-sequence fitting, Cepheid & Supernova parts, etc. was made with the help of Google Gemini.

Let's start by installing important modules:

In [ ]:
!pip install -q astroquery

In [ ]:
from math import *
import numpy as np # this is a math package for python
import matplotlib.pyplot as plt # this is a graphing package for python
from matplotlib import colors
from astroquery.gaia import Gaia # this is our astrophysics package for querying the space mission Gaia
import astropy.units as u # this is a more generic astronomy package
from astropy.coordinates import SkyCoord

## Part I: A color-magnitude diagram for the Hyades cluster

This notebook generates a *color-magnitude diagram* based upon Gaia Early Data Release 3, for detected sources of light found on any patch of the sky, i.e., a *cone search*. The results are filtered on the basis of some data-quality criteria and a user-specified range of proper motions. The user provides the coordinates and radius of the cone search, as well as the data quality control settings, and views two scatter plots of the qualifying sources. Based upon those plots, the user sets the range of proper motions, in right ascension and declination, of sources to be plotted in the cmd.


In [ ]:
# the first step is you need to find the RA and Dec of the Hyades cluster as well as its angular size in the sky.

# Use the following to convert the RA and Dec of the Hyades to decimal values:

print(SkyCoord('4h27m0s', '+15d52m12s', frame='icrs'))

<SkyCoord (ICRS): (ra, dec) in deg
    (66.75, 15.87)>


In [ ]:
# Query GAIA

min_parallax_over_error = 3  # a simple quality-control filter, typically 3 or 6
min_ruwe = 0  # Kuzma et al. 2021 recommend 1.4, but try 0 as well
radius_multiplier = 10
sigmas = 20              # a sort of zoom control on the pm plot -- we need a broad swatch for this

# HERES WHERE YOU ENTER THE TARGET -- som examples below. Note that tradius is in DEGREES

# tname, tra, tdec, tradius = "M 45", 56.3, 25.1, 1
# tname, tra, tdec, tradius = "NGC 3201", 154.4034, -46.41, 40/60
# tname, tra, tdec, tradius = "M 13", 250.423, 36.461, 8/60

tname, tra, tdec, tradius = "Hyades", 66.75, 15.87, 0.5 # Reduced radius further to 0.5 degrees to avoid Gaia server errors

max_sources = 800000
needed_columns =["source_id",
                 "ra","dec",
                 "parallax",  # added
                 "parallax_error",
                 "parallax_over_error",
                 "ruwe",
                 "pmra",
                 "pmdec",
                 "pm",
                 "phot_g_mean_mag",
                 "bp_rp",]

coord = SkyCoord(ra=tra, dec=tdec, unit=(u.degree, u.degree), frame='icrs')
search_radius = u.Quantity(tradius, u.deg)

Gaia.ROW_LIMIT = max_sources
Gaia.MAIN_GAIA_TABLE = 'gaiadr3.gaia_source'

# Added TOP limit and IS NOT NULL filters to prevent the Gaia Java backend from crashing
adql_query = f"""SELECT TOP {max_sources}
  {', '.join(needed_columns)}
FROM
  gaiadr3.gaia_source
WHERE
  1=CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {tra}, {tdec}, {search_radius.value}))
  AND parallax IS NOT NULL
  AND pmra IS NOT NULL
  AND pmdec IS NOT NULL
"""

# Switch to synchronous job to bypass the broken async queue
job = Gaia.launch_job(adql_query)
results = job.get_results()


if min_ruwe == 0:
  filtered_rows = (results["parallax_over_error"] >= min_parallax_over_error)
else:
  filtered_rows = ((results["parallax_over_error"] >= min_parallax_over_error) & \
                     (results["ruwe"] >= min_ruwe))

filtered_results = results[filtered_rows]

print(tname)
print("ra:", tra)
print("dec:", tdec)
print("radius:", search_radius)
print(len(results), "total sources retrieved")
print(len(filtered_results), "sources with parallax_over_error >=", min_parallax_over_error, "and ruwe >=", min_ruwe)


500 Error 500:
Cannot invoke "java.util.List.iterator()" because "results" is null


HTTPError: Error 500:
Cannot invoke "java.util.List.iterator()" because "results" is null

In [ ]:
sigmas


20

##Most of the stars in this data set are NOT in the Hyades cluster.

We need to find the stars that are moving together with the cluster -- look for the smaller cluster of stars with large proper motion in the +ra direction. Hyades is kind of zooming!

In [ ]:
  # plot pm
  fig, ax = plt.subplots(figsize=(6,6), dpi = 100 )

  x1 = filtered_results["pmra"]
  y1 = filtered_results["pmdec"]
  x_mean  = np.mean(x1)
  x_sigma = np.std(x1)
  y_mean  = np.mean(y1)
  y_sigma = np.std(y1)
  ax.scatter(x1, y1, alpha=0.1, s=10, color='b', zorder=0)
  xlim = ax.set_xlim([floor(x_mean-sigmas*x_sigma), ceil(x_mean+sigmas*x_sigma)])
  ylim = ax.set_ylim([floor(y_mean-sigmas*y_sigma), ceil(y_mean+sigmas*y_sigma)])
  ax.set_xlabel("pmra")
  ax.set_ylabel("pmdec")
  ax.set_title(tname + "\n (" + str(len(filtered_results)) \
                + " sources" \
                + ", radius = " + str(search_radius)
                + ", poe >= " + str(min_parallax_over_error) \
                + ", ruwe >= " + str(min_ruwe) \
                + ")")
  ax.minorticks_on()
  ax.grid(which='minor', linestyle=':', linewidth='0.5', color='black')
  ax.grid(which='major', linestyle='-', linewidth='0.5', color='black')


##Filter by proper motion so we just have the Hyades cluster

Exclude all sources but those with proper motions within a specified range and produce a color-magnitude diagram of the resulting subset:

In [ ]:
min_pmra  = 90 # you enter this by hand here - again, look for the small bump to the side
max_pmra  = 110 # ^^
min_pmdec = -40 # ^^
max_pmdec = -20 # ^^

sel_pm = (filtered_results["pmra"]  >= min_pmra)  & \
         (filtered_results["pmra"]  <= max_pmra)  & \
         (filtered_results["pmdec"] <= max_pmdec) & \
         (filtered_results["pmdec"] <= max_pmdec)
plot_subset = filtered_results[sel_pm]
print(len(plot_subset))


# plot the color-magnitude diagram
print(len(plot_subset))
if solicit_pm_ranges:
  range_tag = "pm-selected"
else:
  range_tag = "not pm-selected"

x = plot_subset["bp_rp"]
y = plot_subset["phot_g_mean_mag"]
fig, ax = plt.subplots(figsize=(6, 6), dpi = 100)
ax.scatter(x, y, alpha=0.1, s=10, color='b', zorder=0)
ax.invert_yaxis()
ax.set_xlabel("bp_rp")
ax.set_ylabel("g_mag")
ax.set_title(tname + "\n (" + str(len(plot_subset)) \
                + " sources" \
                + ", radius = " + str(round(search_radius,2))
                + ", poe >= " + str(min_parallax_over_error) \
                + ", ruwe >= " + str(min_ruwe) \
                + ", " + range_tag \
                + ")")
ax.grid(which='major', linestyle='-', linewidth='0.2', color='black')
plt.savefig("gaia-cmd-dr2.png", dpi=140)

## Part II: From a color-(apparent) magnitude diagram to color-(absolute) magnitude diagram

The chart above has apparent magnitude on the vertical axis. We can now use parallax to get from apparent magnitude to absolute magnitude.

Look at line #19 - does that equation look familiar? "np" is Numpy, a math package for python that allows us to compute logarithms well.

In [ ]:
# 1. Filter out invalid parallaxes (must be > 0 to calculate distance)
valid_parallax_mask = filtered_results["parallax"] > 0
valid_stars = filtered_results[valid_parallax_mask]

# 2. Calculate distance in parsecs
# GAIA parallax is given in milliarcseconds (mas), so d = 1000 / p
distance_pc = 1000.0 / valid_stars["parallax"]

# 3. Filter for Hyades cluster members based on distance
# The Hyades cluster is roughly 47 pc away, so we can set a window around that (e.g., 35 to 60 pc)
distance_mask = (distance_pc >= 35) & (distance_pc <= 60)
hyades_stars = valid_stars[distance_mask]

# Get the final distances for the filtered stars
d = 1000.0 / hyades_stars["parallax"]

# 4. Calculate Absolute Magnitude (M = m - 5*log10(d) + 5)
# 'phot_g_mean_mag' is the GAIA apparent G-band magnitude
absolute_mag = hyades_stars["phot_g_mean_mag"] - 5 * np.log10(d) + 5

# Extract the color index for the x-axis
color_index = hyades_stars["bp_rp"]

print(f"Stars remaining after distance filtering: {len(hyades_stars)}")

# 5. Plot the Absolute Magnitude Diagram (Color-Magnitude Diagram)
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ax.scatter(color_index, absolute_mag, alpha=0.6, s=5, color='black')

# Invert the y-axis (brighter stars have lower magnitudes)
ax.invert_yaxis()

ax.set_xlabel("Color Index (BP - RP)")
ax.set_ylabel("Absolute Magnitude (M_G)")
ax.set_title(f"Color-Absolute Magnitude Diagram for {tname} \n(Distance Filtered: 35pc - 60pc)")
ax.grid(True, alpha=0.3)
plt.show()

##Part III: Main-sequence Fitting

We will now compare the Hyades cluster to another cluster (NGC 7790)

https://en.wikipedia.org/wiki/NGC_7790

In [ ]:
from astroquery.gaia import Gaia
from astropy.coordinates import SkyCoord
import astropy.units as u

# Define NGC 7790 coordinates
print("Resolving coordinates for NGC 7790...")
ngc7790_coord = SkyCoord.from_name("NGC 7790")

# Query GAIA DR3 for stars within 0.3 degrees of NGC 7790
# We are grabbing the G-band magnitude (apparent) and the BP-RP color index
print(f"Querying GAIA around RA: {ngc7790_coord.ra.deg:.2f}, Dec: {ngc7790_coord.dec.deg:.2f}...")

query = f"""
SELECT source_id, ra, dec, parallax, pmra, pmdec, phot_g_mean_mag, bp_rp, ruwe
FROM gaiadr3.gaia_source
WHERE 1=CONTAINS(
  POINT('ICRS', ra, dec),
  CIRCLE('ICRS', {ngc7790_coord.ra.deg}, {ngc7790_coord.dec.deg}, 0.2)
)
AND phot_g_mean_mag < 24
AND bp_rp IS NOT NULL
"""

# Launch the query
job = Gaia.launch_job_async(query)
ngc7790_results = job.get_results()

# Convert to a pandas dataframe
ngc7790_df = ngc7790_results.to_pandas()
print(f"Success! Retrieved {len(ngc7790_df)} stars in the direction of NGC 7790.")

In [ ]:
# Aggressive Filtering ---
import matplotlib.pyplot as plt

# Drop rows missing proper motion or parallax data
ngc7790_pm_clean = ngc7790_df.dropna(subset=['pmra', 'pmdec', 'parallax'])

# Proper Motion Filter
# Centering tightly around NGC 7790's true motion of roughly (-2.3, -1.4)
pm_ra_min, pm_ra_max = -8, 4
pm_dec_min, pm_dec_max = -5, 3

# PARALLAX Filter
# NGC 7790 is at a parallax of ~0.3 mas. We will grab a tight slice around that.
plx_min, plx_max = 0.0, 0.7

# Apply both filters!
strict_mask = (
    (ngc7790_pm_clean['pmra'] >= pm_ra_min) & (ngc7790_pm_clean['pmra'] <= pm_ra_max) &
    (ngc7790_pm_clean['pmdec'] >= pm_dec_min) & (ngc7790_pm_clean['pmdec'] <= pm_dec_max) &
    (ngc7790_pm_clean['parallax'] >= plx_min) & (ngc7790_pm_clean['parallax'] <= plx_max)
)

# Apply the strict mask AND drop any stars missing photometry
ngc7790_final_clean = ngc7790_pm_clean[strict_mask].dropna(subset=['phot_g_mean_mag', 'bp_rp'])

# Extract our final cleaned axes for the Main-Sequence fitting cell
ngc7790_color = ngc7790_final_clean['bp_rp']
ngc7790_app_mag = ngc7790_final_clean['phot_g_mean_mag']

print(f"Original star count: {len(ngc7790_df)}")
print(f"Stars remaining for NGC 7790 fitting after filter: {len(ngc7790_final_clean)}")

# Plot the new, zoomed-in Vector Point Diagram to confirm we isolated the cluster
fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
ax.scatter(ngc7790_pm_clean['pmra'], ngc7790_pm_clean['pmdec'], s=1, color='gray', alpha=0.1, label='Discarded Background')
ax.scatter(ngc7790_final_clean['pmra'], ngc7790_final_clean['pmdec'], s=5, color='red', alpha=0.8, label='NGC 7790 Cluster Members')

ax.set_xlabel("Proper Motion RA (mas/yr)")
ax.set_ylabel("Proper Motion Dec (mas/yr)")
ax.set_title("Proper Motion Field (Zoomed In)")
ax.set_xlim(-10, 6)
ax.set_ylim(-6, 6)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Dust-Corrected Main Sequence Fitting ---
import matplotlib.pyplot as plt

# ====================================================================
# ADJUST THIS VALUES TO ALIGN THE NGC 7790 MAIN SEQUENCE WITH HYADES
apparent_distance_modulus = 13.2   # Shifts NGC 7790 Up/Down (Distance + Dust Dimming)

# DO NOT adjust the color_excess (reddening)
color_excess = 0.54               # Shifts NGC 7790 Left/Right (Dust Reddening)
# ====================================================================

# 1. Apply RUWE filter to clean up the messy lower sequence
ngc_super_clean = ngc7790_final_clean[ngc7790_final_clean['ruwe'] < 1.4] # (Change variable name if you updated it!)

# Extract the final, super-clean axes
ngc_app_mag = ngc_super_clean['phot_g_mean_mag']
ngc_color = ngc_super_clean['bp_rp']

# 2. Apply our visual shifts for the graph
ngc_abs_mag_guess = ngc_app_mag - apparent_distance_modulus
ngc_color_unreddened = ngc_color - color_excess

# Set up the plot
fig, ax = plt.subplots(figsize=(10, 8), dpi=100)

# Plot Hyades
ax.scatter(color_index, absolute_mag, alpha=0.5, s=10, color='blue', label='Hyades (Known)')

# Plot NGC 7790
ax.scatter(ngc_color_unreddened, ngc_abs_mag_guess, alpha=0.6, s=10, color='red',
           label=f'NGC 7790 ($\u03bc_{{app}}$={apparent_distance_modulus}, E(BP-RP)={color_excess})')

ax.invert_yaxis()
ax.set_xlabel("Intrinsic Color Index (BP - RP)")
ax.set_ylabel("Absolute Magnitude ($M_G$)")
ax.set_title("Main Sequence Fitting: Hyades vs NGC 7790")
ax.legend()
ax.grid(True, alpha=0.3)

# View limits
ax.set_xlim(-0.5, 3.5)
ax.set_ylim(12, -4)
plt.show()

# Calculate Extinction (A_G) caused by dust
extinction_Ag = 2.0 * color_excess

# Subtract the Extinction from the apparent shift to find the TRUE shift
true_distance_modulus = apparent_distance_modulus - extinction_Ag

# 3. Calculate distance based on the TRUE distance modulus
estimated_distance_pc = 10 ** ((true_distance_modulus + 5) / 5)

print("-" * 50)
print(f"Apparent Distance Modulus:    {apparent_distance_modulus}")
print(f"Color Excess (Reddening):     {color_excess}")
print(f"Extinction Ag (Dimming):      {extinction_Ag:.2f} magnitudes")
print(f"True Distance Modulus:        {true_distance_modulus:.2f}")
print(f"Estimated Distance:           {estimated_distance_pc:,.0f} parsecs")
print("-" * 50)

## Part IV: Investigating Cepheid Variable Stars in NGC 7790

You might have noticed some giant stars in the data set above. There are three classic Cepheid variables among them - CF Cas, CEa Cas, and CEb Cas. We can be boring and just get their periods and apparent magnitudes directly ...

In [ ]:
# Finding the Cepheids in NGC 7790 ---

print(f"Searching for Cepheid variables in NGC 7790...")

# Query the variable star table within our cluster radius
# We are grabbing the period (pf) and the average G-band magnitude
query = f"""
SELECT s.source_id, s.ra, s.dec, v.pf AS period_days, v.int_average_g AS mean_app_mag
FROM gaiadr3.gaia_source AS s
JOIN gaiadr3.vari_cepheid AS v ON s.source_id = v.source_id
WHERE 1=CONTAINS(
  POINT('ICRS', s.ra, s.dec),
  CIRCLE('ICRS', {ngc7790_coord.ra.deg}, {ngc7790_coord.dec.deg}, 0.15)
)
"""

job = Gaia.launch_job_async(query)
ngc_cepheids = job.get_results().to_pandas()

print(f"SUCCESS! Found {len(ngc_cepheids)} Cepheid variables in this cluster.")
display(ngc_cepheids[['source_id', 'period_days', 'mean_app_mag']])

But honestly, I'd like us to check at least one of the light curves to be sure it is right.

In [ ]:
# Grab the Source ID of the first Cepheid in our list (e.g., CF Cas)
target_source_id = ngc_cepheids['source_id'].iloc[0]

print(f"Downloading Epoch Photometry for Source ID: {target_source_id}...")

# Download the time-series data
datalink = Gaia.load_data(ids=[target_source_id],
                          retrieval_type='EPOCH_PHOTOMETRY',
                          data_release='Gaia DR3',
                          data_structure='INDIVIDUAL')

dl_key = list(datalink.keys())[0]
time_series = datalink[dl_key][0].to_table().to_pandas()

# Filter using the newly discovered column names
# We want the G-band time, G-band mag, and we reject bad data flags
g_band_clean = time_series[time_series['rejected_by_photometry'] == False]

# Extract the specific time and magnitude columns for the G-band
obs_time = g_band_clean['g_transit_time']
obs_mag = g_band_clean['g_transit_mag']

print(f"Success! Retrieved {len(obs_time)} individual brightness measurements.")

# Plot the raw, unfolded light curve
fig, ax = plt.subplots(figsize=(10, 4), dpi=100)
ax.scatter(obs_time, obs_mag, color='black', s=10, alpha=0.7)
ax.invert_yaxis() # Brighter is lower!
ax.set_xlabel("Time (Barycentric Julian Date in Days)")
ax.set_ylabel("Apparent Magnitude ($m_G$)")
ax.set_title("Raw, Unfolded GAIA Observations")
ax.grid(True, alpha=0.3)
plt.show()

#### Why does the raw data above look like a mess?
Space telescopes like GAIA don't record continuous video. Instead, as GAIA spins and orbits, it takes random "snapshots" of stars every few weeks. If you plot these random snapshots over the span of a thousand days, it just looks like scattered noise.

#### What is Phase Folding?
If a star's brightness fluctuates randomly, there is no way to organize the dots. But a Cepheid variable star is precise.

When you use the slider to guess a period (let's say, 4.87 days), the computer does a mathematical trick called Modulo Division. It takes the timestamp of every single snapshot GAIA ever took and divides it by 4.87. It throws away the quotient and only looks at the remainder (the phase).

A remainder of 0.0 means the photo was taken exactly at the start of a cycle. A remainder of 0.5 would be halfway through the cycle, etc.

If your guessed period is exactly right, every snapshot taken across years of observation perfectly stacks on top of each other, revealing the true shape of the star's light curve.

#### What are you actually looking at?
When the dots snap into that clean "sawtooth" wave, you are looking at the physical pulsation of the surface of the star due to helium gas ionization.

#### Why does this matter?
Over a century ago, astronomer Henrietta Swan Leavitt discovered that the period of a Cepheid's heartbeat is directly linked to its true, absolute luminosity (The Leavitt Law). The longer it takes to pulse, the physically brighter the star is. By measuring the period from this graph, we can calculate exactly how bright this star really is, and therefore, exactly how far away it must be.

Use the interactive slider below to find the exact period.


In [ ]:
# --- Interactive Phase Folding ---
import ipywidgets as widgets
from IPython.display import display

print("INSTRUCTIONS: Drag the slider until the dots align into a clean, repeating wave pattern. It'll look GOOOOD.\n")

def plot_folded_lightcurve(guessed_period):
    plt.figure(figsize=(10, 5), dpi=100)

    # The Math: Modulo division to calculate the phase (from 0.0 to 1.0)
    # We use our specific G-band obs_time here
    phase = (obs_time % guessed_period) / guessed_period

    # We plot the data twice (from Phase 0 to 2) so you can clearly see the continuous wave
    plt.scatter(phase, obs_mag, color='dodgerblue', alpha=0.8, s=20)
    plt.scatter(phase + 1, obs_mag, color='dodgerblue', alpha=0.8, s=20)

    plt.gca().invert_yaxis()
    plt.xlabel("Phase (Cycles)")
    plt.ylabel("Apparent Magnitude ($m_G$)")
    plt.title(f"Phase-Folded Light Curve \nImplied Period: {guessed_period:.3f} days")
    plt.xlim(0, 2)
    plt.grid(True, alpha=0.3)
    plt.show()

# Create the interactive slider
period_slider = widgets.FloatSlider(
    value=4.8, # Starting near the real value
    min=4.0,
    max=5.2,
    step=0.01,
    description='Period (days):',
    layout=widgets.Layout(width='600px')
)

# Link the slider to the graphing function
widgets.interact(plot_folded_lightcurve, guessed_period=period_slider);

# Plotting period-luminosity

First we will get the periods and luminosities into proper units, then we will make a plot.

In [ ]:
# Calibrating Absolute Magnitudes

# We use the variables we already figured out earlier at the end of Part III

true_distance_modulus = 12.12 # fill this in
extinction_Ag = 0.54 # fill this in

# Calculate the True Absolute Magnitude for all three Cepheids, taking into account distance and extinction by dust
# Formula: M = m - true_distance_modulus - Extinction
ngc_cepheids['abs_mag'] = ngc_cepheids['mean_app_mag'] - true_distance_modulus - extinction_Ag

# For the Leavitt Law, we plot the base-10 logarithm of the period
ngc_cepheids['log_period'] = np.log10(ngc_cepheids['period_days'])

print("Calibrated Cepheids:")
for index, star in ngc_cepheids.iterrows():
    print(f"Period: {star['period_days']:.2f} days | Log(P): {star['log_period']:.3f} | Absolute Mag (M_G): {star['abs_mag']:.2f}")

In [ ]:
# The Period-Luminosity Diagram (Leavitt Law) ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6), dpi=100)

# Plot our three calibrated Cepheids
ax.scatter(ngc_cepheids['log_period'], ngc_cepheids['abs_mag'],
           color='gold', edgecolor='black', s=150, marker='*', zorder=5,
           label='NGC 7790 Cepheids (Calibrated)')

# Set up the axes - you might have to adjust these!
# Y-axis is still inverted from before
ax.set_xlim(0.5, 0.9)
ax.set_ylim(0, -5)
# ax.invert_yaxis()

ax.set_xlabel("Log10(Period in days)")
ax.set_ylabel("Absolute Magnitude ($M_G$)")
ax.set_title("The Leavitt Law (Period-Luminosity Relationship)")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

# Part V: Make the jump to the Large Magellanic Cloud

Honestly, with just 3 stars, we don't really have a "law" just some points. We'd have to do main sequence fitting of dozens of more star clusters to get a good sample set. But the Large Magellanic Cloud is a great source of Cepheids. And because it's big AND far enough away, we can get a lot of Cepheids. Of course, we'll just plot apparent magnitude on the vertical axis. (For now!)

In [ ]:
# The Large Magellanic Cloud (Finding the Slope of the Leavitt Low) ---
from astroquery.gaia import Gaia
import matplotlib.pyplot as plt
import numpy as np

print("Querying GAIA for Cepheids in the Large Magellanic Cloud...")
print("This is a big query, so it might take 15-30 seconds...")

# The LMC is huge, so we search a wide 5-degree radius
# We specifically ask for Classical Cepheids ('DCEP')
query = """
SELECT v.pf AS period_days, v.int_average_g AS app_mag
FROM gaiadr3.vari_cepheid AS v
JOIN gaiadr3.gaia_source AS s ON v.source_id = s.source_id
WHERE 1=CONTAINS(POINT('ICRS', s.ra, s.dec), CIRCLE('ICRS', 80.89, -69.75, 5.0))
AND v.type_best_classification = 'DCEP'
"""

job = Gaia.launch_job_async(query)
lmc_cepheids = job.get_results().to_pandas()

# Calculate Log10 of the periods for the x-axis
lmc_cepheids['log_period'] = np.log10(lmc_cepheids['period_days'])

print(f"SUCCESS! Found {len(lmc_cepheids)} Cepheids in the LMC.")

# Plot the LMC Cepheids
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

ax.scatter(lmc_cepheids['log_period'], lmc_cepheids['app_mag'],
           color='purple', s=2, alpha=0.3, label='LMC Cepheids (Uncalibrated)')

ax.invert_yaxis() # Apparent magnitudes: Brighter is lower
ax.set_xlabel("Log10(Period in days)")
ax.set_ylabel("Apparent Magnitude ($m_G$)")
ax.set_title("The Leavitt Law Slope in the LMC")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()

In [ ]:
# The Fully Calibrated Leavitt Law ---
import matplotlib.pyplot as plt
import numpy as np

lmc_clean = lmc_cepheids.dropna(subset=['log_period', 'app_mag'])

print("Calculating the cosmic shift...")

# Find the slope of the LMC data using a linear regression (line of best fit)
# We use polyfit to get the slope (m) and y-intercept (b) of the Apparent magnitudes
slope, intercept = np.polyfit(lmc_clean['log_period'], lmc_clean['app_mag'], 1)

# Find the average position of our 3  anchor points from NGC 7790
avg_ngc_log_p = ngc_cepheids['log_period'].mean()
avg_ngc_abs_mag = ngc_cepheids['abs_mag'].mean()

# Calculate what the apparent magnitude of our average star WOULD be if it were in the LMC
lmc_expected_app_mag = (slope * avg_ngc_log_p) + intercept

# Calculate the shift (Distance Modulus) required to align the LMC with NGC 7790
# Shift = Apparent Mag - Absolute Mag
lmc_distance_modulus = lmc_expected_app_mag - avg_ngc_abs_mag

print("-" * 50)
print(f"Calculated Slope of Leavitt Law: {slope:.2f}")
print(f"Calculated Distance Modulus to the LMC: {lmc_distance_modulus:.2f}")
print("-" * 50)

# 5. Apply the shift! Convert all LMC Apparent Magnitudes to True Absolute Magnitudes
lmc_cepheids['abs_mag'] = lmc_cepheids['app_mag'] - lmc_distance_modulus

# 6. Plot the Final Masterpiece!
fig, ax = plt.subplots(figsize=(10, 7), dpi=120)

# Plot the calibrated LMC stars
ax.scatter(lmc_cepheids['log_period'], lmc_cepheids['abs_mag'],
           color='purple', s=2, alpha=0.3, label='LMC Cepheids (Calibrated)')

# Plot the line of best fit (The Leavitt Law)
x_line = np.linspace(0.2, 1.6, 100)
y_line = (slope * x_line) + intercept - lmc_distance_modulus
ax.plot(x_line, y_line, color='black', linewidth=2, linestyle='--', label='Leavitt Law Trendline')

# Plot our golden anchor points from NGC 7790
ax.scatter(ngc_cepheids['log_period'], ngc_cepheids['abs_mag'],
           color='gold', edgecolor='black', s=250, marker='*', zorder=5,
           label='NGC 7790 Anchors')

# Set up the axes
ax.invert_yaxis() # Absolute magnitudes: Brighter is lower!
ax.set_xlabel("Log10(Period in days)", fontsize=12)
ax.set_ylabel("Absolute Magnitude ($M_G$)", fontsize=12)
ax.set_title("The Calibrated Cosmic Distance Ladder\n(The Leavitt Law)", fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Calculate the calibrated y-intercept (Absolute Magnitude)
calibrated_intercept = intercept - lmc_distance_modulus

# Create the equation string
equation_text = f"Best Fit Line:\n$M_G = {slope:.2f} \\times \\log_{{10}}(P) {calibrated_intercept:+.2f}$"

# Place the text box in the bottom left corner (0.05, 0.05)
ax.text(0.05, 0.05, equation_text, transform=ax.transAxes, fontsize=12,
        verticalalignment='bottom',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='black', alpha=0.9))

plt.show()

# The final mind-blowing calculation
lmc_distance_pc = 10 ** ((lmc_distance_modulus + 5) / 5)
print(f"\nBased on this calibration, the Large Magellanic Cloud is {lmc_distance_pc:,.0f} parsecs away.\nBut even better than that, we now have a way to find the distance to any galaxy that has a Cepheid variable star in it!")

# Part VI: Type Ia Supernovae

Now we need to find a galaxy that has both Cepheid data and also Type Ia Supernova data. Enter M101, the stunning Pinwheel Galaxy.

https://en.wikipedia.org/wiki/Pinwheel_Galaxy

There are lots of Cepheids in M101, but we can't get them from Gaia, which is too broad a survey. So we're going to query the Hubble Space Telescope dataset, Vizier!

In [ ]:
#  Bridging the Gap from Hubble Cepheids to Supernovae ---
from astroquery.vizier import Vizier # this allows us to query HST data
import ipywidgets as widgets
from IPython.display import display

print("Querying VizieR for Hubble Space Telescope Cepheids in M101...")

# Pull the Hubble Cepheid catalog for M101 (Shappee & Stanek, 2011)
viz = Vizier(columns=['Per', 'Vmag'])
viz.ROW_LIMIT = -1
m101_catalogs = viz.get_catalogs("J/ApJ/733/124/cepheids")
m101_cepheids = m101_catalogs[0].to_pandas()

# Calculate Log Period
m101_cepheids['log_period'] = np.log10(m101_cepheids['Per'])

print("SUCCESS! Data loaded. Use the slider to align the M101 stars with the Leavitt Law!")

# --- The Interactive Plotting Function ---
def fit_m101(distance_modulus_guess):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

    # 1. Plot the Master Calibrated Leavitt Law (The Anchor Line)
    # Using the slope and intercept you calculated in Cell 8
    x_line = np.linspace(0.5, 2.0, 100)
    y_line = (slope * x_line) + calibrated_intercept
    ax.plot(x_line, y_line, color='black', linewidth=3, linestyle='--',
            label='Leavitt Law from the LMC (Absolute Mag)')

    # 2. Shift the M101 Apparent Magnitudes by the guessed Distance Modulus
    shifted_mags = m101_cepheids['Vmag'] - distance_modulus_guess

    # Plot the shifted M101 stars
    ax.scatter(m101_cepheids['log_period'], shifted_mags,
               color='cyan', edgecolor='blue', s=20, alpha=0.6, label='M101 Cepheids')

    # Lock the axes so the graph doesn't bounce while sliding
    # (Setting bottom to 0 and top to -8 naturally inverts the y-axis for magnitudes!)
    ax.set_ylim(0, -8)
    ax.set_xlim(0.4, 2.2)

    ax.set_xlabel("Log10(Period in days)", fontsize=12)
    ax.set_ylabel("Calculated Absolute Magnitude ($M_V$)", fontsize=12)
    # Calculate the physical distance in parsecs based on the slider
    distance_pc = 10 ** ((distance_modulus_guess + 5) / 5)

    # Replace the old ax.set_title with these two lines:
    ax.set_title("Sliding M101 to the Leavitt Law", fontsize=14, fontweight='bold', pad=25)
    ax.text(0.5, 1.02, f"Guessed Distance Modulus: {distance_modulus_guess:.2f}  |  Calculated Distance: {distance_pc:,.0f} parsecs",
            transform=ax.transAxes, ha='center', va='bottom', fontsize=11, color='dimgrey')

    ax.legend(loc='lower left')
    ax.grid(True, alpha=0.3)
    plt.show()

# --- Create and display the slider ---
# M101 is around 29.1, so we give them a range to play with
dm_slider = widgets.FloatSlider(
    value=27.8,
    min=26.0,
    max=31.0,
    step=0.1,
    description='Dist Modulus:',
    layout={'width': '600px'}
)

widgets.interact(fit_m101, distance_modulus_guess=dm_slider);

Now we can apply that distance modulus to the supernova light curve seen in M101

In [ ]:
# Measuring the Supernova Peak (Interactive!) ---

import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Grab the distance modulus dynamically from the previous slider
student_m101_distance = dm_slider.value

print(f"\n\nApplying your M101 Distance Modulus of {student_m101_distance:.2f}...\n\n")

# Real historical photometric data for SN 2011fe
sn_data = {
    'days_from_peak': [-14, -10, -5, -2, 0, 2, 5, 10, 15, 20, 30, 40, 50],
    'app_mag': [13.7, 11.5, 10.2, 9.95, 9.9, 9.95, 10.1, 10.5, 11.0, 11.5, 12.4, 13.2, 14.0]
}
sn_df = pd.DataFrame(sn_data)

# Calculate the Absolute Magnitude using the student's M101 distance!
sn_df['abs_mag'] = sn_df['app_mag'] - student_m101_distance

# --- The Interactive Plotting Function ---
def measure_peak(peak_guess):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

    # Plot the Light Curve
    ax.plot(sn_df['days_from_peak'], sn_df['abs_mag'], marker='o', color='red',
            linestyle='-', linewidth=2, markersize=8, label='SN 2011fe')

    # Add the interactive horizontal ruler
    ax.axhline(peak_guess, color='black', linestyle='--', linewidth=2, label='Your Ruler')
    ax.text(15, peak_guess - 0.2, f"Measured Peak Absolute Mag: {peak_guess:.2f}",
            fontsize=12, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor='black', boxstyle='round,pad=0.4'))

    # Lock the axes so the graph doesn't bounce!
    # (Notice we invert the y-axis by placing the smaller negative number at the top)
    ax.set_ylim(-14.0, -19.0)
    ax.set_xlim(-16, 55)

    ax.set_xlabel("Days from Peak Brightness", fontsize=12)
    ax.set_ylabel("Absolute Magnitude ($M_V$)", fontsize=12)

    ax.set_title("Measuring a Standard Candle", fontsize=14, fontweight='bold', pad=25)
    ax.text(0.5, 1.02, "Slide the ruler to touch the peak of the supernova light curve",
            transform=ax.transAxes, ha='center', va='bottom', fontsize=11, color='dimgrey')

    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')
    plt.show()

# --- Create and display the slider ---
# We start the slider off-target so they have to hunt for it!
peak_slider = widgets.FloatSlider(
    value=-17.0,
    min=-21.0,
    max=-16.0,
    step=0.05,
    description='Measure Peak:',
    layout={'width': '600px'}
)

widgets.interact(measure_peak, peak_guess=peak_slider);

# Part VII: Galactic Redshift

Astronomer Edwin Hubble used the 100" Hooker Telescope on Mt. Wilson near Pasadena, CA, to measure the redshifts of galaxies.  

https://en.wikipedia.org/wiki/Edwin_Hubble

We're going to look for the redshift of UGC 9391:

https://en.wikipedia.org/wiki/UGC_9391

In [ ]:
# Live Query to the Sloan Digital Sky Survey ---
# import numpy as np

from astroquery.sdss import SDSS
from astropy import coordinates as coords
import astropy.units as u

galaxy_name = "UGC 9391"

print("\nAiming virtual telescope at distant galaxy " + galaxy_name)
print("\n(Actually, I'm just querying the Sloan Digital Sky Survey database...)\n\n")

pos = coords.SkyCoord.from_name(galaxy_name)
xid = SDSS.query_region(pos, spectro=True, radius=1*u.arcmin)

if xid is not None:
    print(f"Target Acquired! Downloading the spectrum...")
    sp = SDSS.get_spectra(matches=xid[0:1]) # xid[0:1] keeps it as an astropy Table
    spec_data = sp[0][1].data

    # Save the data to variables that the next cell can see!
    sdss_wavelengths = 10 ** spec_data['loglam']
    sdss_flux = spec_data['flux']

    print("SUCCESS: Real spectral data loaded into memory.")
else:
    print("ERROR: No spectrum found for this galaxy.")

In [ ]:
# Measuring Redshift via Spectroscopy---
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# The speed of light in km/s
C = 299792.458

# Laboratory (Rest-Frame) wavelengths of common emission lines in Angstroms

lab_lines = {
    # --- BLUE END: Mostly Absorption (Stars) ---
    'Ca II K': 3933.7,       # Deep absorption dip
    'Ca II H': 3968.5,       # Deep absorption dip
    'G-band': 4304.4,        # Broad absorption feature
    'H-beta (Peak)': 4861.3, # Common emission/absorption

    # --- GREEN/YELLOW: The "Big Three" ---
    'O III (Peak 1)': 4958.9, # Bright Oxygen emission
    'O III (Peak 2)': 5006.8, # Usually the brightest green peak
    'Mg b (Dip)': 5175.4,     # Deep Magnesium absorption valley
    'Na D (Dip)': 5892.5,     # The "Sodium Streetlight" valley

    # --- RED END: The Emission Forest ---
    'H-alpha (Big Peak)': 6562.8, # The "King" of emission lines
    'N II (Nitrogen)': 6583.4,    # Often a shoulder on H-alpha
    'S II (Peak 1)': 6716.4,      # Sulfur emission doublet
    'S II (Peak 2)': 6730.8       # Sulfur emission doublet
}

# --- The Optimized Interactive Plotting Function ---
def fit_spectrum(z_guess):
    fig, ax = plt.subplots(figsize=(12, 6), dpi=100)

    # Plot the real observed galaxy spectrum
    ax.plot(sdss_wavelengths, sdss_flux, color='black', linewidth=1, label='Observed Spectrum')

    # Sort the lines by wavelength so the staggering logic works left-to-right
    sorted_lines = sorted(lab_lines.items(), key=lambda x: x[1])

    # We will cycle through these 3 heights to prevent overlapping
    heights = [0.96, 0.88, 0.80]

    for i, (name, lab_wave) in enumerate(sorted_lines):
        # The Redshift Equation: Observed = Rest * (1 + z)
        shifted_wave = lab_wave * (1 + z_guess)

        # Draw the vertical line
        ax.axvline(shifted_wave, color='darkorange', linestyle='--', linewidth=1.5, alpha=0.6)

        # Pick a height based on the index (0, 1, or 2)
        y_pos = heights[i % len(heights)]

        # Add the text label with a small 'bbox' (background box) for readability
        ax.text(shifted_wave, np.max(sdss_flux) * y_pos, name,
                rotation=90, verticalalignment='top', horizontalalignment='center',
                fontsize=8, color='darkorange', fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=1))

    # Set the visible window
    ax.set_xlim(4500, 7000)

    # Give the plot some "headroom" for the staggered labels (1.3x max flux)
    ax.set_ylim(0, np.max(sdss_flux) * 1.3)

    ax.set_xlabel("Wavelength (Angstroms $\\AA$)", fontsize=12)
    ax.set_ylabel("Relative Flux", fontsize=12)

    velocity = z_guess * C

    ax.set_title(f"Spectroscopy: Measuring Redshift", fontsize=14, fontweight='bold', pad=25)
    ax.text(0.5, 1.02, f"Guessed Redshift (z): {z_guess:.5f}  |  Calculated Velocity: {velocity:,.0f} km/s",
            transform=ax.transAxes, ha='center', va='bottom', fontsize=12, color='darkred', fontweight='bold')

    ax.grid(True, alpha=0.2)
    plt.show()

# --- Create and display the slider ---
z_slider = widgets.FloatSlider(
    value=0.00000,
    min=0.00000,
    max=0.03000,
    step=0.00050,
    description='Redshift (z):',
    layout={'width': '800px'},
    readout_format='.4f'
)

print("Slide the redshift (z) until your dashed lab lines match the absorption lines of the galaxy.")
widgets.interact(fit_spectrum, z_guess=z_slider);

What happens if we plot redshift vs. distance for a number of galaxies?

In [ ]:
# Calculating the Hubble Constant ---
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Grab the calibrated standard candle from the student's previous slider
student_peak_mag = peak_slider.value

print(f"\n\nUsing your calibrated Standard Candle for Type Ia Supernovae (M_V = {student_peak_mag:.2f})...\n")
print("\nCalculating distances to galaxies in the deep universe...\n\n")

# A curated dataset of Type Ia Supernovae in distant galaxies
# We have their peak apparent magnitudes and their recessional velocities (from redshift)
cosmic_data = {
    'galaxy': ['UGC 9391', 'NGC 1309', 'NGC 3021', 'NGC 3370', 'NGC 1015', 'NGC 1316', 'NGC 1448'],
    'app_peak_mag': [14.3, 14.8, 15.4, 15.8, 16.5, 17.3, 18.2],
    'velocity_kms': [1900, 2400, 3200, 4100, 5800, 8500, 13000] # km/s
}
cosmology_df = pd.DataFrame(cosmic_data)

# 1. Calculate the Distance Modulus for each galaxy
cosmology_df['distance_modulus'] = cosmology_df['app_peak_mag'] - student_peak_mag

# 2. Convert Distance Modulus to True Distance in Parsecs (d = 10^((m-M+5)/5))
cosmology_df['distance_pc'] = 10 ** ((cosmology_df['distance_modulus'] + 5) / 5)

# 3. Convert Parsecs to Megaparsecs (Millions of parsecs) for convenience
cosmology_df['distance_Mpc'] = cosmology_df['distance_pc'] / 1_000_000

# --- The Interactive Plotting Function ---
def fit_hubble(hubble_constant_guess):
    fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

    # Plot the galaxies
    ax.scatter(cosmology_df['distance_Mpc'], cosmology_df['velocity_kms'],
               color='orange', edgecolor='red', s=80, zorder=5, label='Distant Galaxies (Supernovae)')

    # Plot the student's guessed Hubble Law Line (y = mx, where m is H0)
    # x ranges from 0 to 200 Megaparsecs
    x_line = np.linspace(0, 200, 100)
    y_line = hubble_constant_guess * x_line
    ax.plot(x_line, y_line, color='black', linestyle='--', linewidth=2,
            label=f'Your Hubble Law ($H_0$ = {hubble_constant_guess})')

    # Lock the axes so the graph is stable
    ax.set_xlim(0, 200)
    ax.set_ylim(0, 15000)

    ax.set_xlabel("Distance (Megaparsecs)", fontsize=12)
    ax.set_ylabel("Recessional Velocity (km/s)", fontsize=12)

    # Let's calculate the Age of the Universe based on their slope!
    # Age = 1 / H0 (after converting units)
    # 1 Mpc = 3.086e19 km. 1 year = 3.154e7 seconds.
    age_of_universe_years = (1 / hubble_constant_guess) * (3.086e19 / 3.154e7) / 1e9

    ax.set_title("Hubble's Law: The Expansion of the Universe", fontsize=14, fontweight='bold', pad=25)
    ax.text(0.5, 1.02, f"Guessed Expansion Rate: {hubble_constant_guess} km/s/Mpc  |  Calculated Universe Age: {age_of_universe_years:.1f} Billion Years",
            transform=ax.transAxes, ha='center', va='bottom', fontsize=11, color='dimgrey')

    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper left')
    plt.show()

# --- Create and display the slider ---
h0_slider = widgets.IntSlider(
    value=50,
    min=40,
    max=100,
    step=1,
    description='Hubble Const:',
    layout={'width': '600px'}
)

widgets.interact(fit_hubble, hubble_constant_guess=h0_slider);